# 07 — Constrained MPC

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

Notebook 06 showed that naive MPC-lite can degrade performance:

```text
prediction error > control benefit
```

Notebook 07 fixes that by adding:

- bounded command updates
- command regularization
- Δ-control around Kalman estimates
- horizon weighting
- comparison against naive MPC, Kalman, moving average, and oracle

## Main question

Can predictive control help once it is constrained to behave like a stabilizer instead of an aggressive override?

## Key idea

Prediction should refine Kalman feedback, not replace it.

```text
Kalman estimate + small regularized predictive correction
```


## Constrained MPC objective

Let the Kalman command be:

```text
u_base = x_hat
```

Constrained MPC solves for a small correction:

```text
u = u_base + Δu
```

with constraints:

```text
|ΔΩ| ≤ ε_Ω
|ΔB| ≤ ε_B
```

and cost:

```text
future response error
+ λ_step ||Δu||²
+ λ_smooth ||u_k - u_{k-1}||²
```

This prevents the controller from chasing prediction noise.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/constrained_mpc"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_fig(name):
    """Save figure as PNG and PDF using the Notebook 07 naming convention."""
    plt.savefig(f"{FIG_DIR}/07_constrained_mpc_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/07_constrained_mpc_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist


def kalman_filter_joint(z, F, H, Q, R, x0=None, P0=None):
    z = np.asarray(z, dtype=float)
    n_steps = z.shape[0]
    n_state = F.shape[0]

    if x0 is None:
        x = z[0].reshape(n_state, 1)
    else:
        x = np.asarray(x0, dtype=float).reshape(n_state, 1)

    if P0 is None:
        P = np.eye(n_state)
    else:
        P = np.asarray(P0, dtype=float)

    x_filt = np.zeros((n_steps, n_state))
    x_pred = np.zeros((n_steps, n_state))
    P_trace = np.zeros((n_steps, n_state, n_state))
    K_trace = np.zeros((n_steps, n_state, H.shape[0]))

    I = np.eye(n_state)

    for i in range(n_steps):
        x_prior = F @ x
        P_prior = F @ P @ F.T + Q

        y = z[i].reshape(-1, 1) - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)

        x = x_prior + K @ y
        P = (I - K @ H) @ P_prior

        x_pred[i] = x_prior.ravel()
        x_filt[i] = x.ravel()
        P_trace[i] = P
        K_trace[i] = K

    return x_filt, x_pred, P_trace, K_trace


In [ ]:
def estimate_drift_rate(x, window=3):
    x = np.asarray(x, dtype=float)
    rate = np.zeros_like(x)

    for i in range(1, len(x)):
        lo = max(0, i - window)
        xs = np.arange(lo, i + 1)
        ys = x[lo:i + 1]
        if len(xs) >= 2:
            rate[i] = np.polyfit(xs, ys, 1)[0]
        else:
            rate[i] = 0.0

    rate[0] = rate[1] if len(rate) > 1 else 0.0
    return rate


def predict_horizon(current, rate, horizon):
    steps = np.arange(1, horizon + 1)
    return current + steps * rate


def response_error_from_command(delta_omega_future, delta_b_future, command_omega, command_b):
    target = rabi_model(pulse_t, target_A, target_Omega, target_B)
    total = 0.0

    for d_omega, d_b in zip(delta_omega_future, delta_b_future):
        y = rabi_model(
            pulse_t,
            target_A,
            target_Omega + d_omega - command_omega,
            target_B + d_b - command_b,
        )
        total += rmse(y, target)

    return total / len(delta_omega_future)


def naive_mpc_commands(delta_omega_est, delta_b_est, horizon=5, rate_window=3, grid_width_omega=0.08, grid_width_b=0.035, grid_points=31):
    delta_omega_est = np.asarray(delta_omega_est)
    delta_b_est = np.asarray(delta_b_est)

    omega_rate = estimate_drift_rate(delta_omega_est, window=rate_window)
    b_rate = estimate_drift_rate(delta_b_est, window=rate_window)

    omega_cmd = np.zeros_like(delta_omega_est)
    b_cmd = np.zeros_like(delta_b_est)

    for k in range(len(delta_omega_est)):
        pred_omega = predict_horizon(delta_omega_est[k], omega_rate[k], horizon)
        pred_b = predict_horizon(delta_b_est[k], b_rate[k], horizon)

        omega_grid = np.linspace(delta_omega_est[k] - grid_width_omega, delta_omega_est[k] + grid_width_omega, grid_points)
        b_grid = np.linspace(delta_b_est[k] - grid_width_b, delta_b_est[k] + grid_width_b, grid_points)

        best_cost = np.inf
        best_omega = delta_omega_est[k]
        best_b = delta_b_est[k]

        for candidate_omega in omega_grid:
            cost = response_error_from_command(pred_omega, pred_b, candidate_omega, delta_b_est[k])
            if cost < best_cost:
                best_cost = cost
                best_omega = candidate_omega

        best_cost = np.inf
        for candidate_b in b_grid:
            cost = response_error_from_command(pred_omega, pred_b, best_omega, candidate_b)
            if cost < best_cost:
                best_cost = cost
                best_b = candidate_b

        omega_cmd[k] = best_omega
        b_cmd[k] = best_b

    return omega_cmd, b_cmd


def constrained_mpc_commands(
    delta_omega_base,
    delta_b_base,
    horizon=1,
    rate_window=3,
    eps_omega=0.006,
    eps_b=0.003,
    grid_points=17,
    lambda_delta=25.0,
    lambda_smooth=5.0,
    horizon_decay=0.65,
):
    """Regularized Δ-control around a Kalman base command."""
    delta_omega_base = np.asarray(delta_omega_base)
    delta_b_base = np.asarray(delta_b_base)

    omega_rate = estimate_drift_rate(delta_omega_base, window=rate_window)
    b_rate = estimate_drift_rate(delta_b_base, window=rate_window)

    omega_cmd = np.zeros_like(delta_omega_base)
    b_cmd = np.zeros_like(delta_b_base)

    prev_omega = delta_omega_base[0]
    prev_b = delta_b_base[0]

    delta_omega_grid = np.linspace(-eps_omega, eps_omega, grid_points)
    delta_b_grid = np.linspace(-eps_b, eps_b, grid_points)

    target = rabi_model(pulse_t, target_A, target_Omega, target_B)
    weights = horizon_decay ** np.arange(horizon)
    weights = weights / np.sum(weights)

    for k in range(len(delta_omega_base)):
        pred_omega = predict_horizon(delta_omega_base[k], omega_rate[k], horizon)
        pred_b = predict_horizon(delta_b_base[k], b_rate[k], horizon)

        best_cost = np.inf
        best_omega = delta_omega_base[k]
        best_b = delta_b_base[k]

        for d_omega in delta_omega_grid:
            for d_b in delta_b_grid:
                cand_omega = delta_omega_base[k] + d_omega
                cand_b = delta_b_base[k] + d_b

                response_cost = 0.0
                for h, (p_omega, p_b) in enumerate(zip(pred_omega, pred_b)):
                    y = rabi_model(
                        pulse_t,
                        target_A,
                        target_Omega + p_omega - cand_omega,
                        target_B + p_b - cand_b,
                    )
                    response_cost += weights[h] * rmse(y, target)

                delta_cost = lambda_delta * (d_omega**2 + 8.0 * d_b**2)
                smooth_cost = lambda_smooth * ((cand_omega - prev_omega)**2 + 8.0 * (cand_b - prev_b)**2)
                total_cost = response_cost + delta_cost + smooth_cost

                if total_cost < best_cost:
                    best_cost = total_cost
                    best_omega = cand_omega
                    best_b = cand_b

        omega_cmd[k] = best_omega
        b_cmd[k] = best_b
        prev_omega = best_omega
        prev_b = best_b

    return omega_cmd, b_cmd


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_command": np.asarray(delta_omega_hat),
        "b_command": np.asarray(delta_b_hat),
        "omega_controlled": omega_controlled,
        "b_controlled": b_controlled,
        "omega_error": omega_controlled - target_Omega,
        "b_error": b_controlled - target_B,
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "controlled_signals": controlled_signals,
    }


## Synthetic drift environment

Same CGCS control-stack environment as Notebooks 05–06.


In [ ]:
n_blocks = 48
block_times = np.arange(n_blocks)

n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

shared = 0.018 * np.sin(2 * np.pi * block_times / 23 + 0.25)

omega_private = (
    0.050 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.014 * np.sin(2 * np.pi * block_times / 9)
)

b_private = (
    0.0013 * block_times
    + 0.005 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

true_delta_Omega = omega_private + shared
true_delta_B = b_private + 0.22 * shared

Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"n_blocks = {n_blocks}")
print(f"true Ω/B drift correlation = {np.corrcoef(true_delta_Omega, true_delta_B)[0,1]:.3f}")


## Calibration fit per block


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Calibration fit complete.")
print(f"measured Ω/B drift correlation = {np.corrcoef(delta_Omega_est, delta_B_est)[0,1]:.3f}")


## Estimators and policies


In [ ]:
# Baselines
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

ma_window = 4
delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

# Scalar Kalman
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

q_omega = 8.0e-4
q_b = 1.0e-5

delta_Omega_scalar, _, _ = kalman_filter_1d(delta_Omega_est, q=q_omega, r=r_omega, p0=1.0)
delta_B_scalar, _, _ = kalman_filter_1d(delta_B_est, q=q_b, r=r_b, p0=1.0)

# Joint Kalman
z_joint = np.column_stack([delta_Omega_est, delta_B_est])

F = np.eye(2)
H = np.eye(2)

R = np.array([
    [r_omega, 0.0],
    [0.0, r_b],
])

q_cross = 0.35 * np.sqrt(q_omega * q_b)
Q_joint = np.array([
    [q_omega, q_cross],
    [q_cross, q_b],
])

joint_filt, joint_pred, P_joint, K_joint = kalman_filter_joint(
    z_joint,
    F=F,
    H=H,
    Q=Q_joint,
    R=R,
    P0=np.eye(2),
)

delta_Omega_joint = joint_filt[:, 0]
delta_B_joint = joint_filt[:, 1]

# Naive MPC from Notebook 06
naive_horizon = 5
delta_Omega_naive_mpc, delta_B_naive_mpc = naive_mpc_commands(
    delta_Omega_joint,
    delta_B_joint,
    horizon=naive_horizon,
)

# Constrained MPC
constrained_horizon = 1
eps_omega = 0.006
eps_b = 0.003
lambda_delta = 25.0
lambda_smooth = 5.0

delta_Omega_constrained, delta_B_constrained = constrained_mpc_commands(
    delta_Omega_joint,
    delta_B_joint,
    horizon=constrained_horizon,
    eps_omega=eps_omega,
    eps_b=eps_b,
    lambda_delta=lambda_delta,
    lambda_smooth=lambda_smooth,
)

# Oracle
delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print("Policies ready.")


## Command comparison


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Ω drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured Ω drift")
plt.plot(block_times, delta_Omega_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_Omega_naive_mpc, linewidth=2, label="naive MPC")
plt.plot(block_times, delta_Omega_constrained, linewidth=2, label="constrained MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Ω drift / command")
plt.title("Constrained MPC: Ω command comparison")
plt.legend()
plt.tight_layout()

save_fig("omega_command_comparison")
plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true B drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured B drift")
plt.plot(block_times, delta_B_joint, linestyle="-.", linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_B_naive_mpc, linewidth=2, label="naive MPC")
plt.plot(block_times, delta_B_constrained, linewidth=2, label="constrained MPC")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("B drift / command")
plt.title("Constrained MPC: B command comparison")
plt.legend()
plt.tight_layout()

save_fig("offset_command_comparison")
plt.show()


## Evaluate policies


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "joint_kalman": (delta_Omega_joint, delta_B_joint),
    "naive_mpc": (delta_Omega_naive_mpc, delta_B_naive_mpc),
    "constrained_mpc": (delta_Omega_constrained, delta_B_constrained),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}

for name, (delta_omega_hat, delta_b_hat) in policies.items():
    policy_results[name] = evaluate_policy(
        name,
        delta_omega_hat,
        delta_b_hat,
        true_delta_Omega,
        true_delta_B,
    )

for name, result in policy_results.items():
    print(
        f"{name:18s} | "
        f"response RMSE mean = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min phase-lock = {result['min_phase_lock']:.6f}"
    )


## Response-level error comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Constrained MPC: response-level error comparison")
plt.legend()
plt.tight_layout()

save_fig("response_rmse_comparison")
plt.show()


## Worst-case block comparison


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(11, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name, result in policy_results.items():
    plt.plot(pulse_t, result["controlled_signals"][worst_block], linewidth=2, label=name)

plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Constrained MPC: worst-case block {worst_block}")
plt.legend()
plt.tight_layout()

save_fig("worst_case_block_comparison")
plt.show()


## Phase-lock stability


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.ylim(0.68, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Constrained MPC: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()

save_fig("cgcs_stability_comparison")
plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: r["response_rmse_mean"])

policy_table = []
for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": bool(np.all(result["phase_lock"] >= threshold)),
    })

policy_table


In [ ]:
policy_names_ranked = [row["policy"] for row in policy_table]
response_means_ranked = [row["response_rmse_mean"] for row in policy_table]
response_max_ranked = [row["response_rmse_max"] for row in policy_table]

x = np.arange(len(policy_names_ranked))
width = 0.35

plt.figure(figsize=(11, 5))
bars_mean = plt.bar(x - width / 2, response_means_ranked, width, label="mean RMSE")
bars_max = plt.bar(x + width / 2, response_max_ranked, width, label="max RMSE")

plt.xticks(x, policy_names_ranked, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Constrained MPC: policy ranking")
plt.legend()

for bars in [bars_mean, bars_max]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.002,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()

save_fig("policy_ranking_summary")
plt.show()


## Constraint sweep

Sweep the allowed Δ-control radius.

Small bounds can be too conservative.  
Large bounds can recover the failure mode from naive MPC.


In [ ]:
eps_values = [0.001, 0.002, 0.004, 0.006, 0.010, 0.015, 0.025]
eps_sweep_rmse = []

for eps in eps_values:
    cmd_omega, cmd_b = constrained_mpc_commands(
        delta_Omega_joint,
        delta_B_joint,
        horizon=constrained_horizon,
        eps_omega=eps,
        eps_b=eps * 0.5,
        lambda_delta=lambda_delta,
        lambda_smooth=lambda_smooth,
    )

    eval_eps = evaluate_policy(
        f"eps_{eps}",
        cmd_omega,
        cmd_b,
        true_delta_Omega,
        true_delta_B,
    )

    eps_sweep_rmse.append(eval_eps["response_rmse_mean"])

eps_sweep_rmse = np.array(eps_sweep_rmse)
best_eps_idx = int(np.argmin(eps_sweep_rmse))
best_eps = float(eps_values[best_eps_idx])

print(f"Current eps_omega = {eps_omega}")
print(f"Best swept eps_omega = {best_eps}")
print(f"Best swept RMSE = {eps_sweep_rmse[best_eps_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(eps_values, eps_sweep_rmse, marker="o", linewidth=1)
plt.axvline(eps_omega, linestyle="--", linewidth=1, label=f"current eps Ω = {eps_omega}")
plt.axvline(best_eps, linestyle=":", linewidth=1, label=f"best eps Ω = {best_eps}")
plt.xlabel("allowed ΔΩ command radius")
plt.ylabel("mean response RMSE")
plt.title("Constrained MPC: command-bound sweep")
plt.legend()
plt.tight_layout()

save_fig("constraint_sweep")
plt.show()


## Regularization sweep


In [ ]:
lambda_values = [0.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0]
lambda_sweep_rmse = []

for lam in lambda_values:
    cmd_omega, cmd_b = constrained_mpc_commands(
        delta_Omega_joint,
        delta_B_joint,
        horizon=constrained_horizon,
        eps_omega=eps_omega,
        eps_b=eps_b,
        lambda_delta=lam,
        lambda_smooth=lambda_smooth,
    )

    eval_lam = evaluate_policy(
        f"lambda_{lam}",
        cmd_omega,
        cmd_b,
        true_delta_Omega,
        true_delta_B,
    )

    lambda_sweep_rmse.append(eval_lam["response_rmse_mean"])

lambda_sweep_rmse = np.array(lambda_sweep_rmse)
best_lambda_idx = int(np.argmin(lambda_sweep_rmse))
best_lambda = float(lambda_values[best_lambda_idx])

print(f"Current lambda_delta = {lambda_delta}")
print(f"Best swept lambda_delta = {best_lambda}")
print(f"Best swept RMSE = {lambda_sweep_rmse[best_lambda_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(lambda_values, lambda_sweep_rmse, marker="o", linewidth=1)
plt.axvline(lambda_delta, linestyle="--", linewidth=1, label=f"current λ = {lambda_delta}")
plt.axvline(best_lambda, linestyle=":", linewidth=1, label=f"best λ = {best_lambda}")
plt.xlabel("Δ-control regularization λ")
plt.ylabel("mean response RMSE")
plt.title("Constrained MPC: regularization sweep")
plt.legend()
plt.tight_layout()

save_fig("regularization_sweep")
plt.show()


## Save constrained-MPC summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "moving_average_window": int(ma_window),
    "naive_mpc_horizon": int(naive_horizon),
    "constrained_mpc_horizon": int(constrained_horizon),
    "eps_omega": float(eps_omega),
    "eps_b": float(eps_b),
    "lambda_delta": float(lambda_delta),
    "lambda_smooth": float(lambda_smooth),
    "best_eps_omega": float(best_eps),
    "best_eps_response_rmse": float(eps_sweep_rmse[best_eps_idx]),
    "best_lambda_delta": float(best_lambda),
    "best_lambda_response_rmse": float(lambda_sweep_rmse[best_lambda_idx]),
    "q_omega": float(q_omega),
    "q_b": float(q_b),
    "q_cross": float(q_cross),
    "r_omega": float(r_omega),
    "r_b": float(r_b),
    "phase_lock_threshold": float(threshold),
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/constrained_mpc_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/constrained_mpc_summary.json")


## Zip outputs for download from Colab


In [ ]:
!zip -r constrained_mpc_outputs.zip figures results


## Control-stack status

Notebook 07 shows the correct role for predictive control:

```text
not aggressive override
but constrained refinement
```

## Interpretation

If constrained MPC beats naive MPC but does not beat Kalman, the result is still useful:

- Kalman is already near-saturating performance
- constrained MPC avoids pathological over-correction
- prediction becomes safe, even if not always better

## Next directions

- `08_stress_test_fast_drift.ipynb`
- `09_adaptive_constraint_tuning.ipynb`
- `noise-mitigation/01_residual_structure_after_control.ipynb`

Guiding rule:

> Prediction must be bounded by measurement confidence.
